# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abdullah-9862873/FlyRank-Week-1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This notebook picks two findings from the FlyRank research paper and asks where the label comes from and whether the validation design carries the claim. It then re-runs the Week 5 model under a grouped split, runs a leakage audit, and rewrites the boldest claim in safe language.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load `hunting-leakage-and-validating` for this task.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: AI Overviews doubled CTR drop**

The paper observes that pages exposed to AI Overviews show roughly double the CTR decline compared to pages not exposed. This is a directional, cross-sectional observation: the exposure group and the non-exposure group differ in ways beyond just the AI Overview presence.

**Where does the label come from?** CTR is measured as clicks divided by impressions over a fixed window. The "drop" is a comparison between two time periods (before and after AI Overview rollout). The label is the delta in CTR, not a binary decline flag.

**Methodology questions:**
1. How was exposure assigned? If AI Overviews appeared on certain query types (e.g., informational), the exposure group may systematically differ in query intent, which independently affects CTR trends.
2. What is the counterfactual? Without a randomized holdout or a difference-in-differences design, the observed CTR gap could reflect selection bias (queries that trigger AI Overviews may already have different click behavior).
3. Is the "doubled" claim adjusted for query volume, position, or device mix? These confounders shift CTR independently of AI Overview presence.

**Finding 2: Freshness decay**

The paper observes that content freshness (measured as days since last update) correlates with declining search performance. Older content shows higher decline rates.

**Where does the label come from?** Decline is defined as a drop in impressions or clicks between two periods. The label is the direction of the trend, derived from impression counts.

**Methodology questions:**
1. Is the relationship monotonic? Our own signal audit (Week 4) shows the 181-365d bucket declines LESS than the 91-180d bucket (46.7% vs 61.1%). If the paper claims monotonic decay, this non-monotonicity needs explanation.
2. How is freshness measured — content_age_days or days_since_last_update? These are different signals (one is creation age, the other is update recency). The paper should specify which, and whether the result holds for both.
3. Is the freshness effect independent of volume? Our data shows high-impression pages (excellent tier, 46.2% declining) decline less than moderate-impression pages (61.5% declining). If freshness correlates with volume, the decay effect may be partially confounded.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

feature_cols = [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'engaged_sessions_90d',
    'avg_position', 'ctr', 'engagement_rate', 'content_age_days',
    'days_since_last_update', 'word_count'
]
df['has_word_count'] = df['word_count'].notna().astype(int)
df['has_position'] = (df['avg_position'] > 0).astype(int)
feature_cols += ['has_word_count', 'has_position']

X = df[feature_cols].fillna(0)
y = df['is_declining_label']
groups = df['client_id']

print(f'Rows: {len(df):,}, Base rate (declining): {y.mean():.1%}')
print(f'Unique clients: {groups.nunique()}')

Rows: 30,000, Base rate (declining): 54.2%
Unique clients: 32


### Random split (Week 5 baseline)

The Week 5 model used GroupShuffleSplit (80/20, grouped by client). This is already a grouped split, so the "before" number is the grouped result. For comparison, here is a naive random split that ignores client boundaries.

In [2]:
from sklearn.model_selection import train_test_split

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Random split (no grouping)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler_r = StandardScaler()
X_train_r_s = scaler_r.fit_transform(X_train_r)
X_test_r_s = scaler_r.transform(X_test_r)

lr_r = LogisticRegression(random_state=42, max_iter=1000)
lr_r.fit(X_train_r_s, y_train_r)
lr_r_prob = lr_r.predict_proba(X_test_r_s)[:, 1]

rf_r = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf_r.fit(X_train_r, y_train_r)
rf_r_prob = rf_r.predict_proba(X_test_r)[:, 1]

print('=== Random split (ignores client boundaries) ===')
print(f'Train: {len(X_train_r):,}, Test: {len(X_test_r):,}')
print(f'LR  AUC: {roc_auc_score(y_test_r, lr_r_prob):.3f}  F1: {f1_score(y_test_r, lr_r.predict(X_test_r)):.3f}')
print(f'RF  AUC: {roc_auc_score(y_test_r, rf_r_prob):.3f}  F1: {f1_score(y_test_r, rf_r.predict(X_test_r)):.3f}')
print(f'LR  P@10: {precision_at_k(lr_r_prob, y_test_r.values, 10):.2f}  P@20: {precision_at_k(lr_r_prob, y_test_r.values, 20):.2f}')
print(f'RF  P@10: {precision_at_k(rf_r_prob, y_test_r.values, 10):.2f}  P@20: {precision_at_k(rf_r_prob, y_test_r.values, 20):.2f}')

=== Random split (ignores client boundaries) ===
Train: 24,000, Test: 6,000
LR  AUC: 0.655  F1: 0.676
RF  AUC: 0.757  F1: 0.740
LR  P@10: 0.70  P@20: 0.65
RF  P@10: 0.80  P@20: 0.90


### Grouped split (Week 5, honest)

GroupShuffleSplit by client_id ensures no client appears in both train and test.

In [3]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[tr_idx], X.iloc[te_idx]
y_train, y_test = y.iloc[tr_idx], y.iloc[te_idx]

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_s, y_train)
lr_prob = lr.predict_proba(X_test_s)[:, 1]

rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_prob = rf.predict_proba(X_test)[:, 1]

print('=== Grouped split (honest, no client leakage) ===')
print(f'Train: {len(X_train):,} ({groups.iloc[tr_idx].nunique()} clients), Test: {len(X_test):,} ({groups.iloc[te_idx].nunique()} clients)')
print(f'LR  AUC: {roc_auc_score(y_test, lr_prob):.3f}  F1: {f1_score(y_test, lr.predict(X_test)):.3f}')
print(f'RF  AUC: {roc_auc_score(y_test, rf_prob):.3f}  F1: {f1_score(y_test, rf.predict(X_test)):.3f}')
print(f'LR  P@10: {precision_at_k(lr_prob, y_test.values, 10):.2f}  P@20: {precision_at_k(lr_prob, y_test.values, 20):.2f}')
print(f'RF  P@10: {precision_at_k(rf_prob, y_test.values, 10):.2f}  P@20: {precision_at_k(rf_prob, y_test.values, 20):.2f}')

=== Grouped split (honest, no client leakage) ===
Train: 23,837 (25 clients), Test: 6,163 (7 clients)
LR  AUC: 0.573  F1: 0.644
RF  AUC: 0.603  F1: 0.621
LR  P@10: 0.60  P@20: 0.70
RF  P@10: 0.70  P@20: 0.70


### The gap between random and grouped

The gap between random-split and grouped-split scores is itself a finding: it measures how much the model was memorizing client-specific patterns rather than learning generalizable signals.

In [4]:
comparison = pd.DataFrame({
    'split': ['Random', 'Grouped (honest)'],
    'LR_AUC': [
        roc_auc_score(y_test_r, lr_r_prob),
        roc_auc_score(y_test, lr_prob)
    ],
    'RF_AUC': [
        roc_auc_score(y_test_r, rf_r_prob),
        roc_auc_score(y_test, rf_prob)
    ],
    'LR_F1': [
        f1_score(y_test_r, lr_r.predict(X_test_r)),
        f1_score(y_test, lr.predict(X_test))
    ],
    'RF_F1': [
        f1_score(y_test_r, rf_r.predict(X_test_r)),
        f1_score(y_test, rf.predict(X_test))
    ]
})
comparison['LR_AUC_gap'] = comparison['LR_AUC'].diff().abs()
comparison['RF_AUC_gap'] = comparison['RF_AUC'].diff().abs()
print('Gap between random and grouped (positive = memorization was happening):')
comparison

Gap between random and grouped (positive = memorization was happening):


,split,LR_AUC,RF_AUC,LR_F1,RF_F1,LR_AUC_gap,RF_AUC_gap
0,Random,0.654721,0.756863,0.675571,0.740381,NaN,NaN
1,Grouped (honest),0.573134,0.602533,0.644461,0.621469,0.081587,0.15433


### Interpretation

If the random-split AUC is substantially higher than the grouped-split AUC, the model was memorizing client-specific patterns. The grouped number is the honest estimate of how the model performs on unseen clients.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [5]:
print('=== Leakage audit on final feature set ===')
print()

# 1. Label-derived features
label_derived = ['trend_direction', 'trend_pct', 'is_declining_label']
features_used = set(feature_cols)
leaked_label = features_used.intersection(label_derived)
print(f'1. Label-derived features in feature set: {leaked_label if leaked_label else "NONE (clean)"}')
print(f'   Label source: trend_direction == "down"')
print(f'   Forbidden columns: {label_derived}')
print()

# 2. Future/overlapping windows
print('2. Time window check:')
print(f'   Features: 90-day aggregates (impressions_90d, clicks_90d, sessions_90d, etc.)')
print(f'   Label: last_30d vs prev_30d comparison -> trend_direction')
print(f'   The 90-day window INCLUDES the label period (last 30 days).')
print(f'   This is a mild overlap, not a direct leak — the features are aggregates over the full 90d,')
print(f'   while the label measures a directional change within that window.')
print(f'   Severity: LOW. The model sees aggregate performance, not the label itself.')
print()

# 3. Decision-derived features
product_flags = ['health_score', 'needs_ctr_fix', 'needs_refresh', 'action', 'reason_code']
leaked_flags = features_used.intersection(product_flags)
print(f'3. Product flags in feature set: {leaked_flags if leaked_flags else "NONE (clean)"}')
print(f'   Forbidden columns: {product_flags}')
print()

# 4. Population selection check
print('4. Population selection:')
print(f'   No outcome-window filters applied. All 30,000 rows included.')
print(f'   No "active clients only" or "impressions > X" filter that would')
print(f'   silently drop declining pages.')
print()

# 5. Feature importance sanity check
print('5. Feature importance sanity check (RF):')
importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)
print(importances.head(5).to_string(index=False))
print()
top_feat = importances.iloc[0]
if top_feat['importance'] > 0.5:
    print(f'   WARNING: {top_feat["feature"]} has importance {top_feat["importance"]:.3f} (>0.5).')
    print(f'   This could indicate leakage or a dominant feature. Investigate.')
else:
    print(f'   Top feature: {top_feat["feature"]} at {top_feat["importance"]:.3f}. No single feature dominates.')
print()

# Summary
print('=== Summary ===')
print(f'Label-derived: CLEAN')
print(f'Product flags: CLEAN')
print(f'Time window: LOW RISK (90d aggregate overlaps label period, but not a direct leak)')
print(f'Population: CLEAN (no outcome-dependent filtering)')

=== Leakage audit on final feature set ===

1. Label-derived features in feature set: NONE (clean)
   Label source: trend_direction == "down"
   Forbidden columns: ['trend_direction', 'trend_pct', 'is_declining_label']

2. Time window check:
   Features: 90-day aggregates (impressions_90d, clicks_90d, sessions_90d, etc.)
   Label: last_30d vs prev_30d comparison -> trend_direction
   The 90-day window INCLUDES the label period (last 30 days).
   This is a mild overlap, not a direct leak — the features are aggregates over the full 90d,
   while the label measures a directional change within that window.
   Severity: LOW. The model sees aggregate performance, not the label itself.

3. Product flags in feature set: NONE (clean)
   Forbidden columns: ['health_score', 'needs_ctr_fix', 'needs_refresh', 'action', 'reason_code']

4. Population selection:
   No outcome-window filters applied. All 30,000 rows included.
   No "active clients only" or "impressions > X" filter that would
   silentl

### Leakage verdict

The final feature set is clean of label-derived and product-flag leakage. The 90-day aggregate window overlaps the label period (last 30 days), but the features are aggregates over the full 90d, not the label itself. This is a mild, disclosed overlap — not a direct leak. The model cannot read the answer from the features.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [6]:
print('=== Claim rewrite ===')
print()
print('BOLD (original):')
print('  "The baseline rule achieves perfect precision at the top 10,')
print('   proving that stale, high-impression pages always decline."')
print()
print('SAFE (rewritten):')
print('  "In this dataset, the baseline rule (stale >= 180d AND impressions >= 500)')
print('   achieved precision@10 of 1.0 on a client-grouped test set (n=6,163,')
print('   base rate 51.1%). All 10 top-ranked items were declining. This is an')
print('   observed pattern in this portfolio, not a general guarantee. The rule')
print('   is highly selective (17 REFRESH items out of 30,000), which inflates')
print('   precision@k. On broader slices (P@50 = 0.68), performance is closer to')
print('   the base rate. The model and the rule both struggle with moderate-traffic')
print('   pages where decline and stability overlap."')
print()
print('What changed:')
print('  - "proves" -> "observed pattern in this portfolio"')
print('  - "always decline" -> reported the actual precision@50 (0.68) as context')
print('  - Added base rate, sample size, and selectivity caveat')
print('  - Noted the rule is decision-support, not a guarantee')

=== Claim rewrite ===

BOLD (original):
  "The baseline rule achieves perfect precision at the top 10,
   proving that stale, high-impression pages always decline."

SAFE (rewritten):
  "In this dataset, the baseline rule (stale >= 180d AND impressions >= 500)
   achieved precision@10 of 1.0 on a client-grouped test set (n=6,163,
   base rate 51.1%). All 10 top-ranked items were declining. This is an
   observed pattern in this portfolio, not a general guarantee. The rule
   is highly selective (17 REFRESH items out of 30,000), which inflates
   precision@k. On broader slices (P@50 = 0.68), performance is closer to
   the base rate. The model and the rule both struggle with moderate-traffic
   pages where decline and stability overlap."

What changed:
  - "proves" -> "observed pattern in this portfolio"
  - "always decline" -> reported the actual precision@50 (0.68) as context
  - Added base rate, sample size, and selectivity caveat
  - Noted the rule is decision-support, not a guarant

### The claim ladder applied

| Level | Language |
|---|---|
| Observed | "In this dataset, stale + visible pages showed higher decline rates" |
| Measured | "The rule achieved P@10=1.0, P@50=0.68 on a grouped test set" |
| Directional | "Staleness combined with visibility is a signal worth investigating" |
| Decision-support | "These 17 pages look worth reviewing first, because they are stale and still getting impressions" |
| Causal (not supported) | "Updating stale pages will improve their rankings" — we have no intervention data |

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.